<a href="https://colab.research.google.com/github/smerarawal/Smart-Hospital-Appointment-System-with-No-Show-Predictor-and-Reschedule-Generation/blob/main/dbms3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mysql-connector-python sqlalchemy pymysql -q

import pandas as pd, numpy as np
from sqlalchemy import create_engine

DB_HOST="centerbeam.proxy.rlwy.net"; DB_USER="root"; DB_PASSWORD="uqqTtnLRCTBzVhUNDorIPByIknjemOAG"; DB_NAME="railway"; DB_PORT=46001
engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("Connected!")

# ── Load appointments + predictions ───────────────────────────────────────────
appointments = pd.read_sql("SELECT * FROM appointments", engine)
predictions  = pd.read_sql("SELECT * FROM predictions", engine)

# Join in pandas instead of SQL (avoids the appointment_id column issue)
df = appointments.merge(predictions, left_index=True, right_index=True, suffixes=('','_pred'))
df['show_probability'] = 1 - df['noshow_probability']
print(f"Loaded {len(df)} appointments with predictions")

# ── Core formula ───────────────────────────────────────────────────────────────
THRESHOLD = 0.60
CAPACITY  = 1

def overbook_decision(p1, p2, capacity=CAPACITY, threshold=THRESHOLD):
    show1 = 1 - p1; show2 = 1 - p2
    E = show1 + show2
    both_high = (p1 >= threshold) and (p2 >= threshold)
    safe      = (E <= capacity)
    decision  = "ALLOW OVERBOOKING" if both_high and safe else "DENY"
    return {'p1_show':round(show1,3),'p2_show':round(show2,3),'E':round(E,3),'decision':decision}

# ── Worked examples ────────────────────────────────────────────────────────────
print("\n===== OVERBOOKING EXAMPLES =====")
for p1,p2,label in [(0.72,0.78,"Both high risk"),(0.30,0.40,"Both low risk"),(0.85,0.80,"Very high risk"),(0.62,0.65,"Borderline")]:
    r = overbook_decision(p1,p2)
    print(f"\n{label}")
    print(f"  P1 no-show={p1} → show={r['p1_show']}")
    print(f"  P2 no-show={p2} → show={r['p2_show']}")
    print(f"  E[attendance] = {r['p1_show']} + {r['p2_show']} = {r['E']}")
    print(f"  Decision: {'✅' if 'ALLOW' in r['decision'] else '❌'} {r['decision']}")

# ── Apply to today's schedule ──────────────────────────────────────────────────
today = df[df['department']=='Cardiology'].head(20)
rows = []; log = []
for hour, group in today.groupby('hour_of_day'):
    pts = group.reset_index(drop=True)
    if len(pts) >= 2:
        r = overbook_decision(pts.iloc[0]['noshow_probability'], pts.iloc[1]['noshow_probability'])
        rows.append({'Slot':f"{hour:02d}:00",'P1':pts.iloc[0]['noshow_probability'],'P2':pts.iloc[1]['noshow_probability'],'E[attend]':r['E'],'Decision':r['decision']})
        log.append({'slot_time':f"{hour:02d}:00",'department':'Cardiology','patient1_id':int(pts.iloc[0]['patient_id']),'patient2_id':int(pts.iloc[1]['patient_id']),'p1_noshow':pts.iloc[0]['noshow_probability'],'p2_noshow':pts.iloc[1]['noshow_probability'],'expected_attendance':r['E'],'decision':r['decision']})

print("\n===== TODAY'S CARDIOLOGY SCHEDULE =====")
print(pd.DataFrame(rows).to_string(index=False))

if log:
    pd.DataFrame(log).to_sql('schedule_log', engine, if_exists='append', index=False)
    print(f"\nSaved {len(log)} decisions to MySQL schedule_log table")

print("Done! Run Notebook 4 next.")

Connected!
Loaded 1000 appointments with predictions

===== OVERBOOKING EXAMPLES =====

Both high risk
  P1 no-show=0.72 → show=0.28
  P2 no-show=0.78 → show=0.22
  E[attendance] = 0.28 + 0.22 = 0.5
  Decision: ✅ ALLOW OVERBOOKING

Both low risk
  P1 no-show=0.3 → show=0.7
  P2 no-show=0.4 → show=0.6
  E[attendance] = 0.7 + 0.6 = 1.3
  Decision: ❌ DENY

Very high risk
  P1 no-show=0.85 → show=0.15
  P2 no-show=0.8 → show=0.2
  E[attendance] = 0.15 + 0.2 = 0.35
  Decision: ✅ ALLOW OVERBOOKING

Borderline
  P1 no-show=0.62 → show=0.38
  P2 no-show=0.65 → show=0.35
  E[attendance] = 0.38 + 0.35 = 0.73
  Decision: ✅ ALLOW OVERBOOKING

===== TODAY'S CARDIOLOGY SCHEDULE =====
 Slot     P1     P2  E[attend]          Decision
09:00 0.2752 0.5880      1.137              DENY
10:00 0.6651 0.9221      0.413 ALLOW OVERBOOKING
11:00 0.7055 0.6551      0.639 ALLOW OVERBOOKING
14:00 0.2172 0.8805      0.902              DENY
16:00 0.5835 0.8836      0.533              DENY

Saved 5 decisions to MySQL

In [ ]:
!pip install mysql-connector-python sqlalchemy pymysql -q

import pandas as pd, numpy as np
import mysql.connector
from sqlalchemy import create_engine

DB_HOST="centerbeam.proxy.rlwy.net"; DB_USER="root"; DB_PASSWORD="uqqTtnLRCTBzVhUNDorIPByIknjemOAG"; DB_PORT=46001

temp = mysql.connector.connect(host=DB_HOST, user=DB_USER, password=DB_PASSWORD, port=DB_PORT)
temp.cursor().execute("CREATE DATABASE IF NOT EXISTS hospital_db")
temp.close()

DB_NAME="hospital_db"
engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("Connected!")

# Load and merge
appointments = pd.read_sql("SELECT * FROM appointments", engine)
predictions  = pd.read_sql("SELECT * FROM predictions", engine)
df = appointments.merge(predictions, on='appointment_id', suffixes=('','_pred'))
df['show_probability'] = 1 - df['noshow_probability']
print(f"Loaded {len(df)} appointments with predictions")

THRESHOLD=0.60; CAPACITY=1

def overbook_decision(p1, p2, capacity=CAPACITY, threshold=THRESHOLD):
    show1=1-p1; show2=1-p2; E=show1+show2
    both_high=(p1>=threshold) and (p2>=threshold)
    safe=(E<=capacity)
    decision="ALLOW OVERBOOKING" if both_high and safe else "DENY"
    return {'p1_show':round(show1,3),'p2_show':round(show2,3),'E':round(E,3),'decision':decision}

print("\n===== OVERBOOKING EXAMPLES =====")
for p1,p2,label in [(0.72,0.78,"Both high risk"),(0.30,0.40,"Both low risk"),(0.85,0.80,"Very high risk"),(0.62,0.65,"Borderline")]:
    r=overbook_decision(p1,p2)
    print(f"\n{label}")
    print(f"  P1 no-show={p1} → show={r['p1_show']}")
    print(f"  P2 no-show={p2} → show={r['p2_show']}")
    print(f"  E[attendance] = {r['p1_show']} + {r['p2_show']} = {r['E']}")
    print(f"  Decision: {'✅' if 'ALLOW' in r['decision'] else '❌'} {r['decision']}")

today=df[df['department']=='Cardiology'].head(20)
rows=[]; log=[]
for hour,group in today.groupby('hour_of_day'):
    pts=group.reset_index(drop=True)
    if len(pts)>=2:
        r=overbook_decision(pts.iloc[0]['noshow_probability'],pts.iloc[1]['noshow_probability'])
        rows.append({'Slot':f"{hour:02d}:00",'P1':pts.iloc[0]['noshow_probability'],'P2':pts.iloc[1]['noshow_probability'],'E[attend]':r['E'],'Decision':r['decision']})
        log.append({'slot_time':f"{hour:02d}:00",'department':'Cardiology','patient1_id':int(pts.iloc[0]['patient_id']),'patient2_id':int(pts.iloc[1]['patient_id']),'p1_noshow':pts.iloc[0]['noshow_probability'],'p2_noshow':pts.iloc[1]['noshow_probability'],'expected_attendance':r['E'],'decision':r['decision']})

print("\n===== TODAY'S CARDIOLOGY SCHEDULE =====")
print(pd.DataFrame(rows).to_string(index=False))

if log:
    pd.DataFrame(log).to_sql('schedule_log',engine,if_exists='append',index=False)
    print(f"\nSaved {len(log)} decisions to MySQL schedule_log table")

print("Notebook 3 done! Run Notebook 4 next.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.7 MB/s eta 0:00:00
Connected!
Loaded 1003 appointments with predictions

===== OVERBOOKING EXAMPLES =====

Both high risk
  P1 no-show=0.72 → show=0.28
  P2 no-show=0.78 → show=0.22
  E[attendance] = 0.28 + 0.22 = 0.5
  Decision: ✅ ALLOW OVERBOOKING

Both low risk
  P1 no-show=0.3 → show=0.7
  P2 no-show=0.4 → show=0.6
  E[attendance] = 0.7 + 0.6 = 1.3
  Decision: ❌ DENY

Very high risk
  P1 no-show=0.85 → show=0.15
  P2 no-show=0.8 → show=0.2
  E[attendance] = 0.15 + 0.2 = 0.35
  Decision: ✅ ALLOW OVERBOOKING

Borderline
  P1 no-show=0.62 → show=0.38
  P2 no-show=0.65 → show=0.35
  E[attendance] = 0.38 + 0.35 = 0.73
  Decision: ✅ ALLOW OVERBOOKING

===== TODAY'S CARDIOLOGY SCHEDULE =====
 Slot     P1     P2  E[attend]          Decision
09:00 0.8973 0.6901      0.413 ALLOW OVERBOOKING
11:00 0.8511 0.5218      0.627              DENY
14:00 0.8964 0.39